In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "")
dbutils.widgets.text("table_name", "", "")
dbutils.widgets.text("landing_timestamp", "", "")
dbutils.widgets.text("source_file_format", "", "")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
dbutils.fs.ls("/Volumes/adb_caresyncs/landing/landing_volume/neon_postgres/hospitals/2026-11-11_07:34:52/")

In [0]:
# Build the landing path once and reuse it for reading
landing_path = f"/Volumes/adb_caresyncs/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the path (and therefore its parent directories) exists and contains files
path_exists = False
try:
    files = dbutils.fs.ls(landing_path)
    path_exists = len(files) > 0
except Exception:
    path_exists = False

# Exit outside the try block so the exit signal is not swallowed by the except clause
if not path_exists:
    msg = f"No new files received for {source_system}.{table_name} at {landing_path}"
    print(msg)
    dbutils.notebook.exit(msg)

print(f"Landing path found: {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
              .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresyncs.bronze.{table_name}")

In [0]:
%sql
select * from adb_caresyncs.bronze.hospitals